<a href="https://colab.research.google.com/github/www-fidezDAE/fidez.RDS/blob/main/jackson_hararchical_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================================
# UPLOAD EXCEL DATASET
# =========================================================

from google.colab import files

uploaded = files.upload()

Saving agency_banking_data.xlsx to agency_banking_data.xlsx


In [ ]:
# =========================================================
# LOAD DATASET
# =========================================================

import pandas as pd
df = pd.read_excel("agency_banking_data.xlsx")

# View dataset
df.head()

,Security,Accessibility,Transaction_Cost,Technology_Acceptance,SME_Growth
0,3.2,3.7,3.00,3.1,3.38
1,3.5,4.1,2.38,3.6,2.50
2,4.4,4.9,4.63,4.3,3.88
3,4.3,4.6,4.50,3.1,4.63
4,4.2,4.2,3.50,3.6,4.63


In [ ]:
# ==============================
# 1. Import libraries
# ==============================
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ==============================
# 2. Load dataset
# ==============================
df = pd.read_excel("agency_banking_data.xlsx")

# ==============================
# 3. Rename columns (cleaning)
# ==============================
df.columns = [
    "Security",
    "Accessibility",
    "Transaction_Cost",
    "Technology_Acceptance",
    "SME_Growth"
]

# ==============================
# 4. Mean centering variables
# ==============================
df["Security_c"] = df["Security"] - df["Security"].mean()
df["Accessibility_c"] = df["Accessibility"] - df["Accessibility"].mean()
df["TransactionCost_c"] = df["Transaction_Cost"] - df["Transaction_Cost"].mean()
df["TechAccept_c"] = df["Technology_Acceptance"] - df["Technology_Acceptance"].mean()

# ==============================
# 5. Create interaction terms
# ==============================
df["Security_Tech"] = df["Security_c"] * df["TechAccept_c"]
df["Accessibility_Tech"] = df["Accessibility_c"] * df["TechAccept_c"]
df["Cost_Tech"] = df["TransactionCost_c"] * df["TechAccept_c"]

# ==============================
# 6. Define dependent variable
# ==============================
Y = df["SME_Growth"]

# ==============================
# 7. Helper function for regression
# ==============================
def run_model(X, model_name):
    X = sm.add_constant(X)
    model = sm.OLS(Y, X).fit()
    print("\n==============================")
    print(model_name)
    print("==============================")
    print(model.summary())
    return model

# ==============================
# 8. MODEL 1: Main effects only
# ==============================
X1 = df[["Security_c", "Accessibility_c", "TransactionCost_c"]]
model1 = run_model(X1, "MODEL 1: Main Effects")

# ==============================
# 9. MODEL 2: Add moderator
# ==============================
X2 = df[[
    "Security_c",
    "Accessibility_c",
    "TransactionCost_c",
    "TechAccept_c"
]]
model2 = run_model(X2, "MODEL 2: Add Moderator")

# ==============================
# 10. MODEL 3: Add interaction terms
# ==============================
X3 = df[[
    "Security_c",
    "Accessibility_c",
    "TransactionCost_c",
    "TechAccept_c",
    "Security_Tech",
    "Accessibility_Tech",
    "Cost_Tech"
]]
model3 = run_model(X3, "MODEL 3: Moderation Model")

# ==============================
# 11. R-squared change comparison
# ==============================
print("\n==============================")
print("MODEL COMPARISON (R² CHANGE)")
print("==============================")

print(f"Model 1 R²: {model1.rsquared:.4f}")
print(f"Model 2 R²: {model2.rsquared:.4f} | ΔR²: {model2.rsquared - model1.rsquared:.4f}")
print(f"Model 3 R²: {model3.rsquared:.4f} | ΔR²: {model3.rsquared - model2.rsquared:.4f}")

# ==============================
# 12. VIF (multicollinearity check)
# ==============================
X_vif = sm.add_constant(df[[
    "Security_c",
    "Accessibility_c",
    "TransactionCost_c",
    "TechAccept_c",
    "Security_Tech",
    "Accessibility_Tech",
    "Cost_Tech"
]])

vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

print("\n==============================")
print("VARIANCE INFLATION FACTOR (VIF)")
print("==============================")
print(vif_data)


MODEL 1: Main Effects
                            OLS Regression Results                            
Dep. Variable:             SME_Growth   R-squared:                       0.824
Model:                            OLS   Adj. R-squared:                  0.822
Method:                 Least Squares   F-statistic:                     326.7
Date:                Tue, 02 Jun 2026   Prob (F-statistic):           1.32e-78
Time:                        11:18:37   Log-Likelihood:                -69.105
No. Observations:                 213   AIC:                             146.2
Df Residuals:                     209   BIC:                             159.7
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const          